# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

- metoda SOTA (state of the art) - najnowoczesniejsze i najlepsze rozwiazania
- autopep8 do automatycznego formatowania

1. Wszystko w pythonie jest obiektem
2. Funkcje można nestować = zagnieżdzać = funkcja w funkcji
3. Można zwracać funkcje wewnątrz innej funkcji
4. Można przekazywać funkcję jako argument innej funkcji

- Funkcja dekorująca (modifier) - Jest to funkcja, która przyjmuje inną funkcję jako argument i zwraca nową funkcję lub modyfikuje zachowanie funkcji, która została przekazana jako argument. 
- Funkcja dekorowana (original) - Jest to funkcja, której zachowanie jest modyfikowane przez funkcję dekorującą. 
- Funkcja dekorowana jest przekazywana jako argument do funkcji dekorującej.

- *args to zmienne podane jako zwykłe inty lub stringi, zapisywane jako krotka; argumenty pozycyjne
- **kwargs to zmienne podane jako podstawienie do zmiennej np. plec = 'kobieta' i jest przechowywana jako słownik; argumenty nazwane

Łańcuchowanie dekoratorów to technika, która polega na aplikowaniu wielu dekoratorów do jednej funkcji. Można stosować jeden dekorator do funkcji, a następnie stosować kolejny dekorator do wyniku poprzedniego dekorowania.

3 praktyki:
- Enkapsulacja, czyli separacja obowiązków. Funkcja ma robić swoje główne zadanie, a dekorator ma zajmować się czymś „wokół”.
- Ortogonalność
- Możliwość ponownego użycia

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func) #dostep do wiekszej ilosci informacji
    def wrapper(*args, **kwargs): #funkcja do ktorej przekazuje juz argumenty
        start_time = time.perf_counter() #zwraca floata reprezentujacego obecny czas
        result = func(*args, **kwargs) #owrapowana funkcja ktora dostaje argumenty
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer #czyli teraz timer mierzy czas mojej nowej funkcji
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5052 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [ ]:
import functools

# TODO: Implementacja dekoratora
def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in (*args, *kwargs.values()):
            if isinstance(arg, list):
                print(len(arg))
        return func(*args, **kwargs) #oryginalna funkcja z jej parametrami - zwracamy wynik jej dzialania
    return wrapper # - zwracam gotowy wrapper


@show_list_length
def process_data(*args): #czyli robie tak naprawde - show_list_length(process_data)
    pass

process_data([1, 1, 1, 1], 'aaaa', 1, 2, [1,1,1,1,1,1,1])
# wiec drukuje 4 i 7 bo te listy sa znalezione, jakbym cos napisala w process data to bedzie wysweitlone na koncu - z funkcji dekorowanej

4
7
Przetwarzanie


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [ ]:
import functools
from functools import wraps
from datetime import datetime
import time

def logger(filename): # zeby przekazac plik
    def decorator(func): # dekorator, który dostanie funkcje do logowania
        @wraps(func) # zachowuje nazwę funkcji, docstring itd.
        def wrapper(*args, **kwargs): # to co sie zrobi przy wywolaniu
            start = time.time()
            result = func(*args, **kwargs)
            end = time.time()
            duration = end - start
            now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            with open(filename, "a") as f:
                f.write(f"{now} | {func.__name__} | {duration:.4f}s\n")
            
            return result
        return wrapper
    return decorator


In [6]:
@logger("logtest.log")
def test():
    print('Testowanie...')
    time.sleep(1)

test()
test()

Testowanie...
Testowanie...


In [7]:
with open('logtest.log') as f:
    log = f.read()
    print(log)
    f.close()

2026-03-23 11:30:31 | test | 1.0042s
2026-03-23 11:30:32 | test | 1.0008s



### Jak mam o tym myslec?
1. Zaczynam od napisania zwykłej funkcji, która mierzy ten czas i loguje
2. Zamiast wywoływać tą funkcję z argumentem funkcji do mierzenia, zamieniam ją na wrapper
3. Dekorator przyjmuje funkcję, owija ją @wraps, zwraca wrapper
4. Wrapper jest definiowany w dekoratorze i zwraca wynik operacji - to jest moja funkcja początkowa z całą logiką
5. Jeśli mam parametr (tutaj nazwa pliku) to muszę dodać 1 dodatkową warstwę wyżej


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [ ]:
# obiekt, który kontroluje dostęp do atrybutu klasy

class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value): # instancja to konkretny student, value to tworzony email
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value # jesli mail poprawny to zapisuje ręcznie do obiektu - deskryptor nie zrobi tego automatycznie
                                             # uzywam __dict__ bo zwykle self.email znowu skorzystaloby z __set__ robiac petle

class Student:
    email = EmailValidator() # za każdym razem gdy ktoś ustawia student.email, użyj deskryptora
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalskiwsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    # s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Błędny format adresu email: jan.kowalskiwsei.edu.pl


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [ ]:
# TODO: Implementacja deskryptora logującego dostęp
class AccessLogger:
    
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        value = instance.__dict__.get(self.name) # biore ze slownika, ktory przechowuje wartosci
        print(f"GET: {self.name} = {value}")
        return value

    def __set__(self, instance, value):
        print(f"SET: {self.name} = {value}")
        instance.__dict__[self.name] = value # wstawiam do slownik[imie] 


class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

u = Uzytkownik("Marysia", 22)

print(u.imie)
u.wiek = 23
print(u.wiek)

SET: imie = Marysia
SET: wiek = 22
GET: imie = Marysia
Marysia
SET: wiek = 23
GET: wiek = 23
23


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [12]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
# TODO: Implementacja generatora ciągu Collatza
from time import sleep


def collatz_generator(n):
    while n != 1:
        yield n # zwraca pojedynczą wartość do kodu, przetwarzanie w czasie rzeczywistym bez czekania do końca programu
        if n % 2 == 0:
            n = n // 2
            sleep(1) # widac ze liczby pokazuja sie z kazda iteracja, z lista trzeba by bylo czekac na full wynik
        else:
            n = n * 3 + 1
            sleep(1)
    yield 1 # petla zakonczona, zwraca 1

# Test:
for it in collatz_generator(50):
    print(it)

50
25
76
38
19
58
29
88
44
22
11
34
17
52
26
13
40
20
10
5
16
8
4
2
1


---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [27]:
current_user = {"username": "admin", "role": "superuser"}

# TODO: Implementacja dekoratora @require_role
def require_role(role):
    def decorator(func):
        def wrapper(*args, **kwargs):
            if current_user['role'] != role:
                raise PermissionError(f'Rola niezgodna z wymaganiami: {role}.')
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role("superuser")
def delete_user():
    print("Użytkownik usunięty")

@require_role("user")
def view_profile():
    print("Profil użytkownika")

delete_user()
view_profile()

Użytkownik usunięty


PermissionError: Rola niezgodna z wymaganiami: user.

### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
# TODO: Implementacja deskryptora Typed
class Typed:
    
    def __init__(self, type):
        self.type = type

    def __set_name__(self, owner, name): # jak nazywa się atrybut w klasie
        self.priv_name = f"_{name}" # Deskryptor przechowuje logikę sprawdzania typu, ale faktyczna wartość musi być zapisana w instancji, a nie w deskryptorze.
                                    # Prywatna nazwa w instancji pozwala, żeby każdy obiekt miał swoją własną wartość.

    def __get__(self, instance, owner): # instance - konkretna instancja klasy, np. p = Person() 
                                        # owner - klasa, czyli Person.
        if instance is None: # dzieje sie gdy ktoś pyta o atrybut bezpośrednio przez klasę np. Person.age 
                             # w tym przypadku nie ma instancji, więc Python przekazuje instance = None
            return self # chcemy dać dostęp do samego deskryptora, bo getattr dla None zwroci blad
        return getattr(instance, self.priv_name, None)
    
    def __set__(self, instance, value):
        if not isinstance(value, self.type):
            raise TypeError(f'Oczekiwano {self.type}, otrzymano {type(value)}')
        setattr(instance, self.priv_name, value)

In [32]:
class Person:
    name = Typed(str)
    age = Typed(int)

p = Person()
p.name = "Marysia"
p.age = 22 

print(p.name)
print(p.age)

p.age = "22"

Marysia
22


TypeError: Oczekiwano <class 'int'>, otrzymano <class 'str'>

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [46]:
# TODO: Implementacja generatora liczb pierwszych
def prime_generator():
    n = 2
    primes = []
    while True:
        is_prime = True
        for p in primes:
            if p * p > n:
                break
            if n % p == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(n)
            yield n
        n += 1


for p in prime_generator():
    print(p)
    sleep(1)
    if p > 30:
        break

2
3
5
7
11
13
17
19
23
29
31


In [47]:
import itertools

primes_7 = (p for p in prime_generator() if str(p).endswith("7"))

for p in itertools.islice(primes_7, 10):
    print(p)
    sleep(1)

7
17
37
47
67
97
107
127
137
157
